In [1]:
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers
from tnsorflow.keras.layers import Layer
import tensotflow_probabilities as tfp
import albumentations as A

ImportError: cannot import name 'keras' from 'tensorflow' (unknown location)

##### Note That Data Augmentation does not mean dupplicate, It means at each epoch apply some transformatioan randomly on the training data so get new data without storing them which is perfect for memory

#### tf.image flexiple and applied on data pipeline

In [ ]:
def augment(inp):
    image = tf.image.adjust_brightness(inp['image'], delta=0.2)
    image = tf.image.adjust_saturation(image, 2)
    image = tf.image.adjust_gamma(image, gamma=1.4, gain=1.5)
    image = tf.image.flip_left_right(image)
    return {'image': image, 'label':inp['label']}

# dataset.map(augment)

#### keras.layers used inside the model applied on training data ---> random means images changed each epoch

In [ ]:
data_augmentation_layer = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
], name="data_augmentation")

# model = keras.sequential([data_augmentation_layer, ...]) 

#### Custom augmentation based on tf.image function 

In [ ]:
class RotNinety(Layer):
    def __init__(self):
       super().__init__()

    def call(self, image):
       return tf.image.rot90()(image)

#### Mixup training data

In [ ]:
def mixup(data1, data2):
  (image1, label1), (image2, label2) = data1, data2
  lamda = tfp.distributions.beta(0.2, 0.2)
  lamda = lamda.sample(1)[0]

  image = lamda * image1 + (lamda-1) * image2
  label = lamda * tf.cast(label1, dtype=tf.float32) + (lamda-1) * tf.cast(label2, dtype=tf.float32)
  return image, label


tarining_data1 = train_data.shuffle(bufffer_size=8, reshuffle_each_iteration=True).map(resize_rescale)
tarining_data2 = train_data.shuffle(bufffer_size=8, reshuffle_each_iteration=True).map(resize_rescale)

mixed_data = tf.data.Dataset.zip((tarining_data1, tarining_data2))

mixed_data = mixed_data.map(mixup).shuffle().batch()

#### Cutmix 

In [ ]:
def cutmix(data1, data2):
    (image1, label1), (image2, label2) = data1, data2
    H, W, _ = image1.shape
    offset_width, offset_height, target_width, target_height, lamda = bounding_box(Hn W)

    crop_2 = tf.image.crop_to_bounding_box(image2, offset_height=offset_height, offset_width=offset_width, target_height=target_height, target_width=target_width)
    pad_2 = tf.image.pad_to_bounding_box(crop_2, offset_height=50, offset_width=100, target_height=H, target_width=W)

    crop_1 = tf.image.crop_to_bounding_box(image1, offset_height=50, offset_width=100, target_height=100, target_width=100)
    pad_1 = tf.image.crop_to_bounding_box(crop_1, offset_height=50, offset_width=100, target_height=H, target_width=W)

    lamda = 1 - (target_width*target_height)/(H*W)
    label = lamda * tf.cast(label1, dtype=tf.float32) + (lamda-1) * tf.cast(label2, dtype=tf.float32)

    return (image1 - pad_1 + pad_2), label


def bounding_box(H, W):
    lamda = tfp.distributions.beta(0.2, 0.2)
    lamda = lamda.sample(1)[0]

    rx = tf.cast(tfp.distributions.unform(0, W).sample(1)[0], dtype=tf.float32)
    ry = tf.cast(tfp.distributions.unform(0, H).sample(1)[0], dtype=tf.float32)
    
    rw = tf.cast(W * tf.sqrt(1 - lamda), dtype=tf.float32)
    rh = tf.cast(H * tf.sqrt(1 - lamda), dtype=tf.float32)

    rx = tf.clip_by_value(rx - rw//2, 0, W)
    ry = tf.clip_by_value(ry - rh//2, 0, H)

    xb = tf.clip_by_value(rx + rw//2, 0, W)
    yb = tf.clip_by_value(ry + rh//2, 0, H)

    rw = xb - rx
    if(rw==0):
        rw=1

    rh= yb - ry
    if(rh==0):
        rh=1

    return rx, ry, rw, rh, lamda

#### Albumentation

In [ ]:
strong_train_transforms = A.Compose([
    A.RandomResizedCrop(224, 224, scale=(0.6, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.OneOf([
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.2),
        A.ToGray(p=1.0),
    ], p=0.8),
    A.OneOf([
        A.GaussianBlur(blur_limit=(1, 3)),
        A.GaussNoise(var_limit=(10, 50)),
    ], p=0.5),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    A.ToTensorV2(),
])

medical_transforms = A.Compose([
    A.Resize(256, 256),
    A.RandomCrop(224, 224),
    A.SquareSymmetry(p=0.5),  # All 8 rotations/flips - proper for medical data
    A.ElasticTransform(alpha=50, sigma=5, p=0.3),  # Tissue-like distortion
    A.Normalize(mean=[0.5], std=[0.5]),  # Center around 0
    A.ToTensorV2(),
])

# Albumentation Augmentaion
transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.SquareSymmetry(p=0.5),  # All 8 rotations/flips - proper for medical data
    #A.ChannelDropout(channel_drop_range=(1, 2), fill=128, p=0.6),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=1.0),
    ], p=0.6),
    A.OneOf([
        A.GaussianBlur(blur_limit=(1, 3), p=1.0),
        A.GaussNoise(var_limit=(10, 50), p=1.0),
    ], p=0.3),
    A.Resize(224, 224),  
])

def augment_albument_func(image):
    image_np = {'image': image}
    transformed = transforms(**image_np)
    image = transformed['image']
    image = tf.cast(image/255, dtype=tf.float32)
    return image

def augment_albument(inp):
    aug_image = tf.numpy_function(func=augment_albument_func, inp=[inp['image']], Tout=tf.float32)
    aug_image.set_shape((224, 224, 3))
    return aug_image, inp['label']
